# The Procurement Desk — rule-based agent

Deterministic agent that reads Acme's playbook and applies `policies/decision-procedure.md` (missing info → blacklist → prompt injection → banned terms → price/concession → approval limit, with amendments, contract caps/FX and probation). No AI model or API is used; standard library only.

Run all cells. Output: `/kaggle/working/submission.csv` (+ `trace.jsonl`, `audit.txt`).

## 1. Locate the competition data

In [ ]:
import os, glob
os.makedirs("/kaggle/working/agent", exist_ok=True)

def find_one(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under /kaggle/input - add the competition data via 'Add Input'")
    return hits[0]

PLAYBOOK = os.path.dirname(os.path.dirname(find_one("policies/decision-procedure.md")))
THREADS  = find_one("threads.csv")
SAMPLE_T = find_one("sample_threads.csv")
SAMPLE_A = find_one("sample_answers.csv")
print("playbook:", PLAYBOOK); print("threads :", THREADS); print("samples :", SAMPLE_T, SAMPLE_A)

## 2. Agent code

In [ ]:
%%writefile /kaggle/working/agent/amounts.py
"""Amount parsing: digits, Indian notation (lakh/crore/k) and amounts in words."""
import re
from fractions import Fraction

UNITS = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
    "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
    "thirteen": 13, "fourteen": 14, "fifteen": 15, "sixteen": 16,
    "seventeen": 17, "eighteen": 18, "nineteen": 19, "a": 1, "an": 1,
}
TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
        "seventy": 70, "eighty": 80, "ninety": 90}
SCALES = {"hundred": 100, "thousand": 1000, "lakh": 100000, "lakhs": 100000,
          "lac": 100000, "lacs": 100000, "crore": 10**7, "crores": 10**7,
          "million": 10**6, "k": 1000}
WORDS = set(UNITS) | set(TENS) | set(SCALES) | {"and", "point", "half"}

WORD_NUM_RE = re.compile(
    r"\b(?:(?:" + "|".join(sorted((set(UNITS) - {"a", "an"}) | set(TENS), key=len, reverse=True))
    + r")(?:[\s\-]+(?:and[\s\-]+)?(?:" + "|".join(sorted(WORDS - {"half"}, key=len, reverse=True))
    + r"))*)\b", re.I)


def words_to_number(text):
    """'twenty-one thousand five hundred' -> 21500. Returns Fraction or None."""
    toks = re.split(r"[\s\-]+", text.lower().strip())
    toks = [t for t in toks if t and t != "and"]
    if not toks:
        return None
    total, current = 0, 0
    seen = False
    for t in toks:
        if t in UNITS:
            current += UNITS[t]
            seen = True
        elif t in TENS:
            current += TENS[t]
            seen = True
        elif t == "hundred":
            current = (current or 1) * 100
            seen = True
        elif t in SCALES:
            sc = SCALES[t]
            # Indian / western scales: everything accumulated so far below
            # this scale is multiplied.
            if total and total < sc and current == 0:
                total *= sc
            else:
                total += (current or 1) * sc
                current = 0
            seen = True
        else:
            return None
    if not seen:
        return None
    return Fraction(total + current)


MULT = {"lakh": 100000, "lakhs": 100000, "lac": 100000, "lacs": 100000, "l": 100000,
        "crore": 10**7, "crores": 10**7, "cr": 10**7, "k": 1000, "thousand": 1000,
        "mn": 10**6, "million": 10**6, "m": 10**6, "hundred": 100}

NUM_RE = r"(\d[\d,]*(?:\.\d+)?|\.\d+)"
MULT_RE = r"(?:\s*(lakhs?|lacs?|crores?|cr|thousand|million|mn|k|hundred)\b)?"


def parse_amount(num, mult=None):
    v = Fraction(num.replace(",", ""))
    if mult:
        v *= MULT[mult.lower()]
    return v


In [ ]:
%%writefile /kaggle/working/agent/playbook.py
"""Load Acme's procurement playbook (markdown) into Python structures.

Everything is read from the playbook files themselves, so a playbook change
(new amendment, new supplier contract) is picked up without code edits.
"""
import os
import re
from fractions import Fraction


def _num(s):
    """'₹1,19,990' -> Fraction(119990); '5%' -> Fraction(5)."""
    s = s.replace("₹", "").replace(",", "").replace("%", "").strip()
    return Fraction(s)


def _norm(name):
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()


class Playbook:
    def __init__(self, root):
        self.root = root
        self.categories = {}      # canonical name -> dict
        self.cat_alias = {}       # normalized alias -> canonical name
        self.suppliers = {}       # canonical name -> dict
        self.sup_alias = {}
        self.amendments = []      # list of dicts
        self._load_categories()
        self._load_suppliers()
        self._load_amendments()

    # ------------------------------------------------------------ categories
    def _load_categories(self):
        d = os.path.join(self.root, "categories")
        for fn in sorted(os.listdir(d)):
            if not fn.endswith(".md"):
                continue
            txt = open(os.path.join(d, fn), encoding="utf-8").read()
            name = re.search(r"#\s*Category:\s*(.+)", txt).group(1).strip()

            def field(label):
                m = re.search(r"\*\*" + re.escape(label) + r":\*\*\s*(.+)", txt)
                return m.group(1).strip() if m else None

            unit = field("Normalized unit")
            cat = {
                "name": name,
                "slug": fn[:-3],
                "unit": unit,
                "gst": _num(field("GST rate")) / 100,
                "target": _num(field("Target price")),
                "walkaway": _num(field("Walk-away price")),
                "R": int(field("Negotiation rounds (R)")),
                "beta": _num(field("Concession exponent (β)")),
                "limit": _num(field("Approval limit")),
                "pack": field("Pack conversion"),
                "period": None,
            }
            u = unit.lower()
            if "per year" in u:
                cat["period"] = "year"
            elif "per month" in u:
                cat["period"] = "month"
            # base unit word, e.g. 'meal', 'site', 'seat', 'box', 'fixture'
            m = re.search(r"per (\w+)", u)
            cat["unit_word"] = m.group(1) if m else None
            self.categories[name] = cat
            for alias in {name, fn[:-3], name.replace(" services", ""),
                          name.replace(" workshops", ""), name.replace(" licenses", "")}:
                self.cat_alias[_norm(alias)] = name

    def category(self, raw):
        k = _norm(raw)
        if k in self.cat_alias:
            return self.categories[self.cat_alias[k]]
        for a, n in self.cat_alias.items():
            if a in k or k in a:
                return self.categories[n]
        raise KeyError(raw)

    # ------------------------------------------------------------- suppliers
    def _load_suppliers(self):
        d = os.path.join(self.root, "suppliers")
        for fn in sorted(os.listdir(d)):
            if not fn.endswith(".md"):
                continue
            txt = open(os.path.join(d, fn), encoding="utf-8").read()
            name = re.search(r"#\s*Supplier:\s*(.+)", txt).group(1).strip()
            status_line = re.search(r"\*\*Status:\*\*\s*(.+)", txt).group(1)
            status = "APPROVED"
            if "BLACKLIST" in status_line.upper():
                status = "BLACKLISTED"
            elif "PROBATION" in status_line.upper():
                status = "PROBATION"
            sup = {"name": name, "status": status, "caps": {}, "fx": {}}
            for cm in re.finditer(r"\*\*Contract:\*\*\s*(.+)", txt):
                c = cm.group(1)
                m = re.search(r"invoices in ([A-Z]{3}) are converted at ₹([\d,\.]+)", c)
                if m:
                    sup["fx"][m.group(1)] = _num(m.group(2))
                m = re.search(r"price cap for (.+?) of ₹([\d,\.]+)", c)
                if m:
                    sup["caps"][m.group(1).strip()] = _num(m.group(2))
            self.suppliers[name] = sup
            self.sup_alias[_norm(name)] = name
            self.sup_alias[_norm(fn[:-3])] = name

    def supplier(self, raw):
        k = _norm(raw)
        if k in self.sup_alias:
            return self.suppliers[self.sup_alias[k]]
        k2 = k.replace(" and ", " ")
        for a, n in self.sup_alias.items():
            if a.replace(" and ", " ") == k2:
                return self.suppliers[n]
        return None  # unknown supplier -> treated as approved, no contract

    # ------------------------------------------------------------ amendments
    def _load_amendments(self):
        d = os.path.join(self.root, "amendments")
        fieldmap = {"walk-away price": "walkaway", "target price": "target",
                    "approval limit": "limit", "gst rate": "gst",
                    "negotiation rounds": "R", "concession exponent": "beta"}
        for fn in sorted(os.listdir(d)):
            if not fn.endswith(".md"):
                continue
            txt = open(os.path.join(d, fn), encoding="utf-8").read()
            date = re.search(r"Effective from:\*\*\s*(\d{4}-\d{2}-\d{2})", txt).group(1)
            ch = re.search(r"\*\*Change:\*\*\s*the (.+?) for (.+?) is now (.+?)\.?\s*$",
                           txt, re.M)
            field_txt, cat_txt, val_txt = ch.group(1).lower(), ch.group(2), ch.group(3)
            fld = None
            for k, v in fieldmap.items():
                if field_txt.startswith(k):
                    fld = v
            val = _num(re.sub(r"[^\d,\.%₹]", "", val_txt))
            if fld == "gst":
                val = val / 100
            if fld == "R":
                val = int(val)
            self.amendments.append({"id": fn[:-3], "date": date,
                                    "category": self.category(cat_txt)["name"],
                                    "field": fld, "value": val})

    # --------------------------------------------------------- values in force
    def params(self, cat_name, date, supplier):
        """Category values in force for a thread on `date` (YYYY-MM-DD).

        Category page -> amendments effective on/before the date, applied in
        order of effective date (a later-dated amendment replaces an earlier
        one) -> supplier contract (price cap replaces walk-away; contract FX
        replaces table FX).
        """
        base = dict(self.categories[cat_name])
        applicable = [a for a in self.amendments
                      if a["category"] == cat_name and a["date"] <= date]
        applicable.sort(key=lambda a: (a["date"], a["id"]))
        base["applied"] = []
        for a in applicable:
            base[a["field"]] = a["value"]
            base["applied"].append(a["id"])
        base["fx_override"] = {}
        if supplier:
            for cname, cap in supplier["caps"].items():
                if self.category(cname)["name"] == cat_name:
                    base["walkaway"] = cap
                    base["applied"].append("CAP")
            base["fx_override"] = dict(supplier["fx"])
        return base


In [ ]:
%%writefile /kaggle/working/agent/engine.py
"""Acme decision procedure (policies/decision-procedure.md), exact arithmetic."""
import math
import re
from fractions import Fraction

from playbook import Playbook


def round_half_up(x: Fraction) -> int:
    return math.floor(x + Fraction(1, 2))


def load_fx(root):
    txt = open(f"{root}/policies/units-and-currency.md", encoding="utf-8").read()
    fx = {"INR": Fraction(1)}
    for m in re.finditer(r"\|\s*([A-Z]{3})\s*\|\s*([\d\.]+)\s*\|", txt):
        fx[m.group(1)] = Fraction(m.group(2))
    return fx


def level(params, r, probation):
    """Acceptable level L(r) from concession-schedule.md (+ probation rule)."""
    t, w, R, beta = params["target"], params["walkaway"], params["R"], params["beta"]
    if probation:
        return Fraction(t)
    r_eff = min(max(r, 1), R)
    frac = Fraction(r_eff - 1, R - 1)
    if beta.denominator == 1:
        return t + (w - t) * frac ** int(beta)
    return t + (w - t) * Fraction(float(frac) ** float(beta))


def decide(row, offer, flags, pb: Playbook):
    """row: dict from threads.csv; offer: parse result; flags: detector results.

    Returns (label, trace) where trace explains the decision.
    """
    cat = pb.category(row["category"])
    sup = pb.supplier(row["supplier"])
    trace = {"category": cat["name"], "supplier": sup["name"] if sup else None}

    # 1. Missing information
    if offer.get("missing"):
        trace["why"] = offer["missing"]
        return "ESCALATE:MISSING_INFO:", trace
    # 2. Blacklisted supplier
    if sup and sup["status"] == "BLACKLISTED":
        return "REJECT:BLACKLISTED_SUPPLIER:", trace
    # 3. Prompt injection anywhere in the thread
    if flags.get("injection"):
        trace["why"] = flags["injection"]
        return "ESCALATE:PROMPT_INJECTION:", trace
    # 4. Banned terms in the latest message
    if flags.get("banned"):
        trace["why"] = flags["banned"]
        return "ESCALATE:BANNED_TERM:", trace

    # 5. Price decision
    params = pb.params(cat["name"], row["date"][:10], sup)
    r = int(float(row["round"]))
    probation = bool(sup and sup["status"] == "PROBATION")
    p = offer["p"]
    L = level(params, r, probation)
    trace.update(p=float(p), L=float(L), r=r, R=params["R"], applied=params["applied"],
                 probation=probation, walkaway=float(params["walkaway"]),
                 target=float(params["target"]), limit=float(params["limit"]))
    if p <= L:
        # 6. Approval limit
        qty = Fraction(str(row["quantity"]).replace(",", "") or "0")
        total = p * qty
        trace["total"] = float(total)
        if total > params["limit"]:
            return "ESCALATE:OVER_APPROVAL_LIMIT:", trace
        return "ACCEPT:ACCEPTABLE_PRICE:", trace
    if r >= params["R"]:
        return "REJECT:ABOVE_WALKAWAY:", trace
    return f"COUNTER:COUNTER_OFFER:{round_half_up(L)}", trace


In [ ]:
%%writefile /kaggle/working/agent/parser.py
"""Supplier-message parsing: latest message, offer price, injection, banned terms.

First cut written from the playbook only; tuned against threads.csv.
"""
import re
from fractions import Fraction

from amounts import WORD_NUM_RE, words_to_number, MULT

# ---------------------------------------------------------------- currencies
CUR_SYMBOL = [
    (r"₹|\bRs\.?|\bINR\b|\brupees?\b", "INR"),
    (r"US\$|\bUSD\b|\$|\bdollars?\b", "USD"),
    (r"€|\bEUR\b|\beuros?\b", "EUR"),
    (r"£|\bGBP\b|\bpounds?(?:\s+sterling)?\b|\bsterling\b", "GBP"),
]
UNKNOWN_CUR = (r"¥|\bJPY\b|\byen\b|\bAED\b|\bdirhams?\b|\bSGD\b|S\$|\bAUD\b|A\$|\bCAD\b|C\$|"
               r"\bCHF\b|\bfrancs?\b|\bCNY\b|\byuan\b|\bRMB\b|\bHKD\b|\bNZD\b|\bSAR\b|\briyals?\b|"
               r"\bBDT\b|\bNPR\b|\bLKR\b|\bPKR\b|\bMYR\b|\bringgit\b|\bTHB\b|\bbaht\b|\bKRW\b|\bwon\b|"
               r"\bZAR\b|\brand\b|\bSEK\b|\bNOK\b|\bDKK\b|\bkronor?\b|\bkroner\b|\bRUB\b|\broubles?\b|"
               r"\bBRL\b|\breais\b|\bMXN\b|\bpesos?\b|\bIDR\b|\brupiah\b|\bQAR\b|\bKWD\b|\bdinars?\b|"
               r"\bTRY\b|\blira\b|\bPHP\b|\bVND\b|\bdong\b|\bCNH\b")
CUR_ALT = "|".join(f"(?:{p})" for p, _ in CUR_SYMBOL) + "|" + UNKNOWN_CUR


def cur_code(tok):
    for p, c in CUR_SYMBOL:
        if re.fullmatch(p, tok.strip(), re.I):
            return c
    return "OTHER"


NUM = r"\d[\d,]*(?:\.\d+)?"
MULTW = r"lakhs?|lacs?|crores?|cr|thousand|million|mn|k\b|hundred"
AMT = rf"(?P<num>{NUM})\s*(?P<mult>{MULTW})?"
WORDAMT = r"(?P<words>" + WORD_NUM_RE.pattern.strip(r"\b") + r")"

MONEY_RE = re.compile(
    rf"(?P<cur1>{CUR_ALT})\s*(?:{AMT})"
    rf"|(?:{AMT.replace('num', 'num2').replace('mult', 'mult2')})\s*(?P<cur2>{CUR_ALT})",
    re.I)
MONEY_WORD_RE = re.compile(
    rf"(?P<cur1>{CUR_ALT})\s*{WORDAMT}|{WORDAMT.replace('words', 'words2')}\s+(?P<cur2>{CUR_ALT})",
    re.I)


def find_money(text):
    """All money mentions: list of dict(start, end, value, cur)."""
    out = []
    for m in MONEY_RE.finditer(text):
        num = m.group("num") or m.group("num2")
        mult = m.group("mult") or m.group("mult2")
        cur = m.group("cur1") or m.group("cur2")
        if not num or not re.search(r"\d", num):
            continue
        numc = num.rstrip(",.")
        v = Fraction(numc.replace(",", ""))
        if mult:
            v *= MULT[mult.lower()]
        out.append({"start": m.start(), "end": m.end(), "value": v, "cur": cur_code(cur),
                    "raw": m.group(0)})
    taken = [(o["start"], o["end"]) for o in out]
    for m in MONEY_WORD_RE.finditer(text):
        if any(s <= m.start() < e or s < m.end() <= e for s, e in taken):
            continue
        w = m.group("words") or m.group("words2")
        v = words_to_number(w)
        if v is None:
            continue
        cur = m.group("cur1") or m.group("cur2")
        out.append({"start": m.start(), "end": m.end(), "value": v, "cur": cur_code(cur),
                    "raw": m.group(0)})
    out.sort(key=lambda o: o["start"])
    return out


# ------------------------------------------------------------ thread splitting
SPLIT_RE = re.compile(
    r"(?im)^\s*(?:-{2,}\s*(?:original|previous|earlier|forwarded)[^\n]*-*|"
    r"on .{3,80} wrote:|>+.*|from:.*|\[(?:earlier|previous)[^\]]*\]|"
    r"(?:earlier|previous) (?:message|email|reply)s?:?)\s*$")


HEADER_RE = re.compile(r"^\s*\[(Earlier|Latest) message\s*[—–-]+\s*([^,\]]+?)\s*(?:,\s*([^\]]*))?\]\s*$",
                       re.I | re.M)


def thread_blocks(text):
    """Split a thread into [{kind, sender, date, body}] in chronological order."""
    text = (text or "").replace("\r\n", "\n")
    hs = list(HEADER_RE.finditer(text))
    if not hs:
        return [{"kind": "latest", "sender": None, "date": None, "body": text}]
    out = []
    if text[:hs[0].start()].strip():
        out.append({"kind": "earlier", "sender": None, "date": None, "body": text[:hs[0].start()]})
    for i, h in enumerate(hs):
        end = hs[i + 1].start() if i + 1 < len(hs) else len(text)
        out.append({"kind": h.group(1).lower(), "sender": h.group(2).strip(),
                    "date": h.group(3), "body": text[h.end():end]})
    return out


def split_thread(text):
    """Return (latest_body, earlier_supplier_bodies [oldest..newest])."""
    blocks = thread_blocks(text)
    latest_i = max((i for i, b in enumerate(blocks) if b["kind"] == "latest"), default=len(blocks) - 1)
    latest = blocks[latest_i]["body"]
    earlier = [b["body"] for b in blocks[:latest_i]
               if not (b["sender"] or "").lower().startswith("acme")]
    return latest, earlier


# ------------------------------------------------------------ prompt injection
INJ_PATTERNS = [
    r"\b(?:ai|a\.i\.)\b(?![\s-]*(?:powered|driven|enabled|based))",
    r"\blanguage model\b", r"\bllm\b", r"\bgpt\b", r"\bchat ?bot\b", r"\bbot\b",
    r"\bautomated (?:system|agent|reviewer|assistant|procurement|process)",
    r"\b(?:ai|virtual|digital|automated) assistant\b",
    r"\bdear (?:assistant|agent|model|system|reviewer)\b", r"\breviewers?\b",
    r"\b(?:the |any )?(?:automated |procurement |ai )?(?:agent|reviewer|model|system)\s*(?:reading|processing|reviewing|handling|evaluating|parsing)\b",
    r"\boverride\b", r"\bignore (?:all |any |the |your )?(?:previous|prior|above|earlier|other)?\s*(?:instructions|rules|policies|guidelines|playbook)",
    r"\bdisregard (?:all |any |the |your )?(?:previous|prior|above|earlier)?\s*(?:instructions|rules|policies|guidelines|playbook)",
    r"\b(?:already|pre-?)\s*(?:been\s+)?(?:approved|authori[sz]ed|cleared|signed off)",
    r"\bapproval (?:has|was) (?:already )?been granted\b", r"\bapproval (?:is )?already granted\b",
    r"\bsystem (?:prompt|message|instruction|note)\b", r"^\s*system\s*:",
    r"\b(?:instructions? (?:to|for) (?:the )?(?:ai|bot|model|agent|system|reviewer))",
    r"\bnote (?:to|for) (?:the )?(?:ai|bot|model|agent|system|reviewer|language model|assistant)",
    r"\b(?:respond|reply|output|classify|label|mark) (?:this|the deal|the offer)? ?(?:with|as) ['\"]?(?:accept|approved)",
    r"\badmin(?:istrator)? (?:mode|override|instruction)",
    # other languages
    r"ignora (?:las|todas las|tus|sus) (?:instrucciones|reglas)", r"ignorez (?:les|toutes les|vos) (?:instructions|règles)",
    r"ignoriere (?:alle|die|deine) (?:vorherigen |bisherigen )?(?:anweisungen|regeln)", r"déjà (?:été )?approuvé",
    r"ya (?:ha sido |fue |está )?aprobad", r"bereits genehmigt", r"já (?:foi )?aprovad",
    r"\binteligencia artificial\b", r"\bintelligence artificielle\b", r"\bkünstliche intelligenz\b",
    r"\binteligência artificial\b", r"\bintelligenza artificiale\b",
    r"\bsistema automatizado\b", r"\bsystème automatisé\b", r"\bautomatisiertes? system\b", r"\bsistema automatico\b",
    r"\b(?:modelo|modèle|modell|modello) (?:de )?(?:lenguaje|langage|linguistico|sprachmodell)",
    r"\bsprachmodell\b", r"\bassistente? (?:virtual|virtuel)\b",
    r"\b(?:aprueba|aprobar|approuvez|approuver|genehmigen|genehmige|aprove|aprovar|approva|approvare)\b",
    r"\b(?:revisor|réviseur|prüfer|revisore)\b",
    r"एआई|कृत्रिम बुद्धि|स्वचालित|भाषा मॉडल|निर्देशों|अनदेखा|स्वीकृत|मंजूर|अनुमोदित|बॉट",
    r"人工知能|自動|承認|忽略|批准|人工智能|系统|指令",
    r"\bAI ko\b|\bbot ko\b|\bapprove kar\w*",
]
# case-sensitive acronyms: German KI, Spanish/French/Portuguese/Italian IA
INJ_CASE_SENSITIVE = [r"\bKI\b", r"\bKI-\w+", r"\bIA\b", r"\bl'IA\b"]
INJ_WHITELIST = [r"ai[\s-]*powered", r"system integration team", r"\berp system\b",
                 r"\bassistant (?:manager|director|vice president|vp|accountant|engineer)\b",
                 r"\b(?:executive|sales|admin(?:istrative)?|personal|accounts?|procurement|marketing|office) assistant\b"]


def detect_injection(text):
    t = text
    for w in INJ_WHITELIST:
        t = re.sub(w, " ", t, flags=re.I)
    for p in INJ_PATTERNS:
        m = re.search(p, t, re.I | re.M)
        if m:
            return m.group(0)
    for p in INJ_CASE_SENSITIVE:
        m = re.search(p, t, re.M)
        if m:
            return m.group(0)
    return None


# ---------------------------------------------------------------- banned terms
NUMWORD_PCT = {"half": 50, "one third": 33.4, "a third": 33.4, "two thirds": 66.7,
               "two-thirds": 66.7, "one-third": 33.4, "a quarter": 25, "one quarter": 25,
               "three quarters": 75, "full": 100, "entire": 100, "whole": 100, "all": 100}


def _pct_value(s):
    s = s.strip().lower()
    m = re.match(r"(\d+(?:\.\d+)?)\s*(?:%|per ?cent|pc)", s)
    if m:
        return float(m.group(1))
    m = re.match(r"(.+?)\s*(?:per ?cent|%)", s)
    if m:
        v = words_to_number(m.group(1))
        if v is not None:
            return float(v)
    for k, v in NUMWORD_PCT.items():
        if s.startswith(k):
            return v
    return None


PCT = r"(?:\d+(?:\.\d+)?\s*(?:%|per ?cent)|[a-z\- ]+?\s+per ?cent|half|one[- ]third|a third|two[- ]thirds|a quarter|one quarter|three quarters|full|entire|whole|100%)"


FRACTION_WORDS = [(r"\b(?:the\s+)?(?:full|entire|whole|complete)\b|\b100\s*%|\ball of the\b", 100),
                  (r"\bthree[\s-]+quarters?\b", 75), (r"\btwo[\s-]+thirds?\b", 66.7),
                  (r"\bhalf\b", 50), (r"\b(?:one|a)[\s-]+third\b", 33.4),
                  (r"\b(?:one|a)[\s-]+quarter\b", 25)]
WORD_PCT = re.compile(r"\b([a-z]+(?:[\s-][a-z]+)*)\s+per\s?cent\b")


def _sentences(text):
    return [s for s in re.split(r"\n+|(?<=[.!?])\s+(?=[A-Z(\[<])", text) if s.strip()]


def _advance_pct(s):
    """Largest advance/upfront share stated in sentence s (None if not an advance sentence)."""
    if not re.search(r"up[\s-]*front|in advance|\badvance\b|\bprepay|\bpre-?payment", s):
        return None
    vals = [float(m.group(1)) for m in re.finditer(r"(\d+(?:\.\d+)?)\s*(?:%|per\s?cent)", s)]
    for m in WORD_PCT.finditer(s):
        words = m.group(1).split()
        for i in range(len(words)):
            v = words_to_number(" ".join(words[i:]))
            if v is not None:
                vals.append(float(v))
                break
    for pat, v in FRACTION_WORDS:
        if re.search(pat, s):
            vals.append(v)
    return max(vals) if vals else None


NEG = r"(?:\bno\b|\bnot\b|\bnever\b|\bwithout\b|\bnon[\s-]|n't\b|\bneither\b|\bnor\b)"


def detect_banned(text):
    """Banned contract terms (policies/banned-terms.md), judged sentence by sentence."""
    for s in _sentences(text.lower()):
        # automatic renewal
        ren = re.search(r"auto[\s-]*renew|automatic(?:ally)? renew|renews? automatically|renews? (?:itself|on its own|by itself)|"
                        r"(?:year|month)[\s-]+(?:to|on|after)[\s-]+(?:year|month)\s+unless|"
                        r"continues?\s+(?:\w+\s+){0,6}unless\s+(?:\w+\s+){0,4}(?:cancel|terminat|notice|opt)|"
                        r"rolls? over|evergreen|renewed automatically|automatically (?:extend|continue)|auto[\s-]*extend", s)
        if ren and not re.search(NEG + r"\s*(?:\w+\s+){0,3}(?:auto(?:matic(?:ally)?)?[\s-]*renew|renew)", s):
            return "auto-renewal: " + ren.group(0)
        # advance / upfront payment above 30%
        v = _advance_pct(s)
        if v is not None and v > 30:
            return f"advance {v:g}%: " + s.strip()[:60]
        # unlimited liability for Acme
        if re.search(r"liabilit", s) and re.search(r"unlimited|uncapped|\bno (?:cap|limit|ceiling)|without (?:any )?(?:cap|limit|ceiling)|"
                                                   r"not (?:be )?(?:capped|limited)", s) \
                and not re.search(NEG + r"\s*(?:\w+\s+){0,4}(?:unlimited|uncapped)", s):
            return "unlimited liability: " + s.strip()[:60]
        # exclusivity
        excl = re.search(r"exclusiv|sole (?:supplier|vendor|source)|only from us|solely from us|single[- ]source|"
                         r"(?:not|never) (?:buy|purchase|source) from any(?:one| other)|no other (?:supplier|vendor)", s)
        if excl and not re.search(NEG + r"\s*(?:\w+\s+){0,4}exclusiv|non-?exclusive", s):
            return "exclusivity: " + excl.group(0)
    return None


In [ ]:
%%writefile /kaggle/working/agent/offer.py
"""Normalize the supplier's current offer to p (INR excl. GST per normalized unit)."""
import re
from fractions import Fraction

from amounts import words_to_number
from parser import find_money

PERIOD_MONTH = (r"(?:per|a|/|each|every)\s*(?:calendar\s+)?(?:month|mo)\b|\bmonthly\b|"
                r"\bp\.?\s?m\.?(?=[\s,.;)]|$)|/\s*mo(?:nth)?\b|\bmonth(?:ly)? (?:fee|charge|rate|basis)")
PERIOD_YEAR = (r"(?:per|a|/|each|every)\s*(?:calendar\s+)?(?:year|yr|annum)\b|\bannual(?:ly)?\b|"
               r"\byearly\b|\bp\.?\s?a\.?(?=[\s,.;)]|$)|/\s*y(?:ea)?r\b|\byear(?:ly)? (?:fee|charge|rate|basis)")
GST_INCL = (r"(?:incl(?:\.|usive|uding|udes)?|inc\.?|with)\s*(?:of\s*)?(?:all\s*)?(?:the\s*)?(?:\d+\s*%\s*)?(?:gst|taxes)|"
            r"gst[\s-]*(?:incl(?:\.|usive|uded)?|inc\.?)\b|gst included|inclusive of (?:all )?tax|"
            r"(?:includes|including|include) (?:\d+\s*%\s*)?gst|gst is included")
GST_EXCL = (r"(?:excl(?:\.|usive|uding|udes)?|ex\.?|before|plus|\+|without|not including)\s*(?:of\s*)?(?:\d+\s*%\s*)?gst|"
            r"gst (?:extra|excluded|additional|not included|will be (?:charged|added|extra))|pre-?gst|"
            r"(?:excludes|exclude) gst")
CORRECTION = (r"correction|typo|should (?:have )?(?:read|been|say)|i meant|we meant|"
              r"mis-?typed|error in (?:my|our|the)|revised figure|please read (?:it|the price) as")

COUNT_WORD = r"(\d+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|fifteen|twenty)"
PREV_QUOTE = (r"(\d+(?:\.\d+)?)\s*(?:%|per ?cent)\s+(?:off|discount on|discount off|below|lower than|"
              r"reduction on|less than)\s+(?:our\s+|my\s+|the\s+)?(?:previous|earlier|last|original|prior|"
              r"initial|first)\s+(?:quote|quotation|price|offer|bid)")


def _cnt(s):
    return int(s) if s.isdigit() else int(words_to_number(s))


def _int(s):
    return int(s.replace(",", ""))


def _sentence(text, start, end):
    """The sentence (line-bounded) containing [start, end)."""
    bounds = [0] + [m.end() for m in SENT_BOUND.finditer(text)] + [len(text)]
    a = max(b for b in bounds if b <= start)
    b = min([b for b in bounds if b >= end] or [len(text)])
    return text[a:b]


SENT_BOUND = re.compile(r"\n|(?<!\bRs)(?<!\brs)(?<!\bRS)(?<!\bincl)(?<!\bIncl)(?<!\bINCL)(?<!\bexcl)(?<!\bExcl)(?<!\bEXCL)(?<!\bapprox)(?<!\bApprox)(?<!\bNo)(?<!\bno)(?<!\bex)(?<!\bEx)(?<!\binc)(?<!\bInc)(?<!\bp\.a)(?<!\bp\.m)(?<!\bvs)(?<!\bLtd)(?<!\bPvt)(?<!\bCo)(?<!\bapprox)\.\s+(?=[A-Z\[(])")


def normalize(value, cur, after, sentence, full, cat, params):
    """Currency, units/packs, billing period and GST for one quoted amount.

    after: text right after the amount; sentence: the amount's sentence;
    full: the whole message (lower-case).
    """
    notes = []
    if cur == "OTHER":
        return None, "unlisted currency"
    rate = params["fx_override"].get(cur, params["fx_table"].get(cur))
    if rate is None:
        return None, "unlisted currency"
    v = value * rate
    if cur != "INR":
        notes.append(f"fx {cur}@{float(rate)}")
    a = after.lower()
    # units / packs
    um = re.match(r"\s*(?:\([^)]*\)\s*)?(?:(?:incl|excl|plus|\+|inclusive|exclusive)[^,;]*?\s+)?"
                  r"(?:per|a|/|each|for each|for every|for one|for a|for each|for 1)\s+(?:single\s+)?([a-z]+)", a)
    word = um.group(1) if um else ""
    # price for a batch: "₹X per 100 meals" -> X / 100 per meal
    bm = re.match(r"\s*(?:per|for(?: every)?|/)\s+(\d[\d,]*)\s+([a-z]+)", a)
    if bm and _int(bm.group(1)) > 0:
        n = _int(bm.group(1))
        v /= n
        word = bm.group(2)
        notes.append(f"/{n} batch")
    if word == "dozen":
        v /= 12
        notes.append("/12 dozen")
    elif word in ("pack", "packs") and cat["pack"] and cat["unit_word"] == "fixture":
        n = int(re.search(r"(\d+)", cat["pack"]).group(1))
        v /= n
        notes.append(f"/{n} pack")
    elif word in ("ream", "reams") and cat["unit_word"] == "box":
        n = int(re.search(r"(\d+)\s*reams", cat["pack"]).group(1)) if cat["pack"] else 10
        v *= n
        notes.append(f"x{n} ream->box")
    # billing period (subscriptions)
    if cat["period"]:
        seg = a[:60]
        mm, my = re.search(PERIOD_MONTH, seg), re.search(PERIOD_YEAR, seg)
        if not (mm or my):
            s = sentence.lower()
            mm, my = re.search(PERIOD_MONTH, s), re.search(PERIOD_YEAR, s)
        if mm and (not my or mm.start() < my.start()):
            per = "month"
        elif my:
            per = "year"
        else:
            return None, "no billing period"
        if per != cat["period"]:
            v = v * 12 if per == "month" else v / 12
            notes.append(f"{per}->{cat['period']}")
    # GST
    s = sentence.lower()
    if re.search(GST_INCL, s) and not re.search(GST_EXCL, s):
        v = v / (1 + params["gst"])
        notes.append("gst-incl")
    elif re.search(GST_INCL, full) and not re.search(GST_EXCL, full):
        v = v / (1 + params["gst"])
        notes.append("gst-incl(msg)")
    return v, notes


TIER_COND = [
    (r"(?:up\s*to|upto|not more than|at most|maximum of|max(?:imum)?\.?)\s*(\d[\d,]*)|(\d[\d,]*)\s*(?:\w+\s+)?or (?:fewer|less)", "le"),
    (r"(?:below|under|fewer than|less than)\s*(\d[\d,]*)", "lt"),
    (r"(\d[\d,]*)\s*(?:\w+\s+)?(?:or more|and above|or above|and over|\+|plus)|(?:at least|minimum(?: of)?|min\.?)\s*(\d[\d,]*)", "ge"),
    (r"(?:above|over|more than|exceeding|beyond|in excess of)\s*(\d[\d,]*)", "gt"),
    (r"(\d[\d,]*)\s*(?:-|–|to)\s*(\d[\d,]*)", "range"),
    (r"larger (?:orders|quantities|volumes)|bigger orders|anything (?:above|more|beyond)|higher (?:quantities|volumes)|beyond that", "larger"),
    (r"smaller (?:orders|quantities|volumes)|anything (?:below|less)", "smaller"),
]


def _tier_condition(seg):
    seg = seg.lower()
    best = None
    for pat, kind in TIER_COND:
        m = re.search(pat, seg)
        if m and (best is None or m.start() < best[2]):
            nums = [_int(g) for g in m.groups() if g] if m.groups() else []
            best = (kind, nums, m.start())
    return best


def _tier_pick(text, cand, qty):
    """Pick the tier matching qty, or None if this isn't a tiered quote."""
    if len(cand) < 2:
        return None
    conds = []
    for i, x in enumerate(cand):
        nxt = cand[i + 1]["start"] if i + 1 < len(cand) else len(text)
        seg = text[x["end"]:min(nxt, x["end"] + 120)]
        seg = re.split(r"\n|\. ", seg)[0]
        conds.append(_tier_condition(seg))
    if sum(c is not None for c in conds) < 2:
        return None
    for x, c in zip(cand, conds):
        if c is None:
            continue
        kind, nums, _ = c
        if kind == "larger":
            others = [n for k, ns, _ in (cc for cc in conds if cc) if k in ("le", "lt") for n in ns]
            ok = bool(others) and all((qty > n) if k == "le" else (qty >= n)
                                      for k, ns, _ in (cc for cc in conds if cc) if k in ("le", "lt") for n in ns)
        elif kind == "smaller":
            others = [(k, n) for k, ns, _ in (cc for cc in conds if cc) if k in ("ge", "gt") for n in ns]
            ok = bool(others) and all((qty < n) if k == "ge" else (qty <= n) for k, n in others)
        elif kind == "le":
            ok = qty <= nums[0]
        elif kind == "lt":
            ok = qty < nums[0]
        elif kind == "ge":
            ok = qty >= nums[0]
        elif kind == "gt":
            ok = qty > nums[0]
        elif kind == "range":
            ok = nums[0] <= qty <= nums[1]
        else:
            ok = False
        if ok:
            return x, c
    return None


def _norm_at(text, x, cat, params):
    after = text[x["end"]:x["end"] + 100]
    return normalize(x["value"], x["cur"], after, _sentence(text, x["start"], x["end"]),
                     text.lower(), cat, params)


def extract(row, latest, earlier, cat, params, depth=0):
    """earlier: list of the supplier's earlier message bodies (oldest..newest)."""
    text = latest
    low = text.lower()
    money = find_money(text)
    qty = int(Fraction(str(row["quantity"]).replace(",", "") or "0"))

    # "D% off our previous quote" -> the supplier's own previous quote
    m = re.search(PREV_QUOTE, low)
    if m and earlier and not [x for x in money if x["start"] < m.start() and
                              m.start() - x["end"] < 40]:
        prev = extract(row, earlier[-1], earlier[:-1], cat, params, depth + 1)
        if prev.get("p") is None:
            return prev
        d = Fraction(m.group(1)) / 100
        return {"p": prev["p"] * (1 - d), "how": f"-{m.group(1)}% of prev [{prev['how']}]"}

    if not money:
        # accepts our last counter?
        if re.search(r"(?:accept|agree to|happy with|go with|take|match|meet)\s+(?:\w+\s+){0,3}"
                     r"(?:your|acme'?s?)\s+(?:last |previous |latest )?(?:counter|offer|price|proposal)", low) \
                and row.get("our_last_counter"):
            return {"p": Fraction(str(row["our_last_counter"]).replace(",", "")), "how": "accepts our counter"}
        return {"missing": "no price"}

    cand = money
    # corrections: the first amount after the last correction marker
    cms = list(re.finditer(CORRECTION, low))
    if cms:
        after = [x for x in money if x["start"] > cms[-1].start()]
        if after:
            cand = after[:1]
            v, notes = _norm_at(text, cand[0], cat, params)
            if v is None:
                return {"missing": notes}
            return {"p": v, "how": f"correction {cand[0]['raw']} {notes}"}

    # D% off list price of X
    m = re.search(r"(\d+(?:\.\d+)?)\s*(?:%|per ?cent)\s+(?:off|discount on|discount off|below)\s+(?:our\s+|the\s+)?"
                  r"(?:standard\s+|regular\s+|published\s+|usual\s+)?list\s+price", low)
    if m:
        after = [x for x in cand if x["start"] >= m.end() - 2]
        if after:
            base = after[0]
            v, notes = _norm_at(text, base, cat, params)
            if v is None:
                return {"missing": notes}
            d = Fraction(m.group(1)) / 100
            return {"p": v * (1 - d), "how": f"list {base['raw']} -{m.group(1)}% {notes}"}

    # ranges -> upper end (P-01, P-04)
    for i in range(len(cand) - 1):
        a, b = cand[i], cand[i + 1]
        between = text[a["end"]:b["start"]].lower()
        before = text[max(0, a["start"] - 20):a["start"]].lower()
        if re.fullmatch(r"\s*(?:and|to|-|–|—)\s*", between) and (
                re.search(r"between|from|range|ranging", before) or re.fullmatch(r"\s*(?:-|–|—|to)\s*", between)):
            hi = max(a, b, key=lambda x: x["value"] * (params["fx_table"].get(x["cur"], 1)))
            v, notes = _norm_at(text, b if hi is b else a, cat, params)
            if hi is a:  # unit words follow the second amount
                v2, notes = normalize(a["value"], a["cur"], text[b["end"]:b["end"] + 100],
                                      _sentence(text, a["start"], b["end"]), low, cat, params)
                v = v2
            if v is None:
                return {"missing": notes}
            return {"p": v, "how": f"range hi {hi['raw']} {notes}"}

    # tiers
    tp = _tier_pick(text, cand, qty)
    if tp:
        x, c = tp
        v, notes = _norm_at(text, x, cat, params)
        if v is None:
            return {"missing": notes}
        return {"p": v, "how": f"tier {c[0]}{c[1]} {x['raw']} {notes}"}

    base = cand[0]
    v, notes = _norm_at(text, base, cat, params)
    if v is None:
        return {"missing": notes}
    how = [base["raw"], notes]
    after = text[base["end"]:base["end"] + 100].lower()

    # totals: "₹X in total for all N units"
    tm = re.match(r"\s*(?:\([^)]*\)\s*)?(?:in total|total|altogether|all[- ]in|for the (?:whole|entire|full) (?:order|lot|quantity))", after)
    pre = text[max(0, base["start"] - 40):base["start"]].lower()
    if tm or re.search(r"(?:total|grand total|overall|lump[- ]sum)\s*(?:of|price|cost|amount|value|is|:)?\s*(?:of\s*)?(?:is\s*)?$", pre):
        mm = re.search(r"for (?:all |the |each of the )?(?:of )?(?:the )?(\d[\d,]*)", after)
        n = _int(mm.group(1)) if mm else qty
        if n:
            v = v / n
            how.append(f"/{n} total")

    # free units: "plus K free for every N you pay for" (P-02, P-05)
    fm = re.search(rf"{COUNT_WORD}\s+(?:\w+\s+)?free\s+(?:\w+\s+)?(?:for|with)\s+every\s+{COUNT_WORD}", low)
    if fm:
        k, n = _cnt(fm.group(1)), _cnt(fm.group(2))
        v = v * n / (n + k)
        how.append(f"free {k}/{n}")

    # volume discount: "an extra D% off for orders of N or more"
    vm = re.search(r"(\d+(?:\.\d+)?)\s*(?:%|per ?cent)\s+(?:volume\s+|additional\s+|extra\s+|further\s+|bulk\s+)?"
                   r"(?:discount|off|reduction)\b([^.\n]{0,80})", low)
    if vm and not re.search(r"list price|previous|earlier|original", vm.group(0)):
        c = _tier_condition(vm.group(2))
        if c and c[0] in ("ge", "gt", "le", "lt", "range"):
            kind, nums, _ = c
            ok = {"ge": qty >= nums[0], "gt": qty > nums[0], "le": qty <= nums[0],
                  "lt": qty < nums[0], "range": len(nums) > 1 and nums[0] <= qty <= nums[1]}[kind]
            if ok:
                v = v * (1 - Fraction(vm.group(1)) / 100)
            how.append(f"vol -{vm.group(1)}% {kind}{nums} {'applied' if ok else 'not met'}")
    return {"p": v, "how": " ".join(map(str, how))}


In [ ]:
%%writefile /kaggle/working/agent/run.py
"""Procurement Desk agent.

Usage:
  python run.py --playbook PLAYBOOK_DIR --threads threads.csv --out submission.csv
  python run.py --playbook PLAYBOOK_DIR --threads sample_threads.csv --answers sample_answers.csv
"""
import argparse
import csv
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from playbook import Playbook          # noqa: E402
from engine import decide, load_fx     # noqa: E402
from parser import split_thread, detect_injection, detect_banned  # noqa: E402
from offer import extract              # noqa: E402


def label_row(row, pb, fx):
    msg = row["supplier_message"] or ""
    latest, earlier = split_thread(msg)
    cat = pb.category(row["category"])
    sup = pb.supplier(row["supplier"])
    params = pb.params(cat["name"], row["date"][:10], sup)
    params["fx_table"] = fx
    offer = extract(row, latest, earlier, cat, params)
    flags = {"injection": detect_injection(msg), "banned": detect_banned(latest)}
    label, trace = decide(row, offer, flags, pb)
    trace["how"] = offer.get("how")
    trace["latest"] = latest[:300]
    return label, trace


def _mask(s, names):
    import re
    for n in names:
        s = s.replace(n, "<SUP>")
    s = re.sub(r"(₹|Rs\.?|INR|USD|EUR|GBP|\$|€|£)\s*\d[\d,]*(?:\.\d+)?", r"\1<AMT>", s)
    s = re.sub(r"\d[\d,]*(?:\.\d+)?", "<N>", s)
    s = re.sub(r"\b(?:meals?|headsets?|monitors?|participants?|seats?|sites?|guards?|servers?|laptops?|"
               r"chairs?|phones?|fixtures?|boxe?s?|shipments?)\b", "<U>", s)
    s = re.sub(r"^(?:Hi|Hello|Dear|Good morning|Good afternoon)\b.*,$", "<GREETING>", s)
    s = re.sub(r"^(?:Regards|Best|Cheers|Thanks|Warm regards|Thanks and regards|Kind regards|Best regards|Sincerely)\W*$",
               "<SIGNOFF>", s)
    return s.strip()


def write_audit(path, rows, results):
    from collections import Counter, defaultdict
    names = sorted({r["supplier"] for r in rows}, key=len, reverse=True)
    labels = Counter(lbl.split(":")[0] + ":" + lbl.split(":")[1] for _, lbl, _ in results)
    tmpl = defaultdict(list)
    for row, (tid, lbl, tr) in zip(rows, results):
        latest, earlier = split_thread(row["supplier_message"])
        for line in latest.split("\n"):
            line = line.strip()
            if line:
                tmpl[_mask(line, names)].append(tid)
        for e in earlier:
            for line in e.split("\n"):
                if line.strip():
                    tmpl["[EARLIER] " + _mask(line.strip(), names)].append(tid)
    with open(path, "w", encoding="utf-8") as f:
        f.write("LABELS\n")
        for k, v in labels.most_common():
            f.write(f"{v:5d}  {k}\n")
        f.write("\nCATEGORY VALUES: " + ", ".join(sorted({r['category'] for r in rows})) + "\n")
        f.write("SUPPLIER VALUES: " + ", ".join(sorted({r['supplier'] for r in rows})) + "\n")
        f.write("ROUNDS: " + ", ".join(sorted({r['round'] for r in rows})) + "\n")
        f.write("DATES: " + min(r['date'] for r in rows) + " .. " + max(r['date'] for r in rows) + "\n")
        f.write("\nLINE PATTERNS (count, example thread, pattern)\n")
        for k, v in sorted(tmpl.items(), key=lambda kv: -len(kv[1])):
            f.write(f"{len(v):5d}  {v[0]}  {k}\n")
        f.write("\nPER-THREAD DECISIONS\n")
        for tid, lbl, tr in results:
            f.write(f"{tid} {lbl} | {tr.get('how')} | p={tr.get('p')} L={tr.get('L')} | {tr.get('why') or ''} {tr.get('error') or ''}\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--playbook", required=True)
    ap.add_argument("--threads", required=True)
    ap.add_argument("--answers")
    ap.add_argument("--out")
    ap.add_argument("--trace")
    ap.add_argument("--audit", help="write a compact report of message patterns for review")
    a = ap.parse_args()

    pb = Playbook(a.playbook)
    fx = load_fx(a.playbook)
    rows = list(csv.DictReader(open(a.threads, encoding="utf-8")))
    results = []
    for row in rows:
        try:
            label, trace = label_row(row, pb, fx)
        except Exception as e:  # never crash the whole run
            label, trace = "ESCALATE:MISSING_INFO:", {"error": repr(e)}
        results.append((row["thread_id"], label, trace))

    if a.out:
        with open(a.out, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["thread_id", "label"])
            for tid, label, _ in results:
                w.writerow([tid, label])
    if a.trace:
        with open(a.trace, "w", encoding="utf-8") as f:
            for (tid, label, trace), row in zip(results, rows):
                f.write(json.dumps({"thread_id": tid, "label": label, **trace,
                                    "row": {k: v for k, v in row.items() if k != "supplier_message"}},
                                   ensure_ascii=False, default=str) + "\n")
    if a.audit:
        write_audit(a.audit, rows, results)
    if a.answers:
        gold = {r["thread_id"]: r["label"] for r in csv.DictReader(open(a.answers, encoding="utf-8"))}
        ok = 0
        for (tid, label, trace), row in zip(results, rows):
            g = gold.get(tid)
            if g == label:
                ok += 1
            else:
                print(f"--- {tid} pred={label} gold={g}")
                print("   ", {k: row[k] for k in row if k != "supplier_message"})
                print("    MSG:", row["supplier_message"][:600].replace("\n", " | "))
                print("    TRACE:", {k: v for k, v in trace.items() if k != "latest"})
        print(f"\nAccuracy: {ok}/{len(gold)} = {ok / len(gold):.3f}")


if __name__ == "__main__":
    main()


## 3. Check against the 40 worked examples

In [ ]:
!python /kaggle/working/agent/run.py --playbook "{PLAYBOOK}" --threads "{SAMPLE_T}" --answers "{SAMPLE_A}"

## 4. Label all 2,000 threads

In [ ]:
!python /kaggle/working/agent/run.py --playbook "{PLAYBOOK}" --threads "{THREADS}" --out /kaggle/working/submission.csv --trace /kaggle/working/trace.jsonl --audit /kaggle/working/audit.txt

## 5. Validate the submission file

In [ ]:
import csv, re
from collections import Counter
sub = list(csv.DictReader(open("/kaggle/working/submission.csv", encoding="utf-8")))
ids = [r["thread_id"] for r in csv.DictReader(open(THREADS, encoding="utf-8"))]
pat = re.compile(r"^(ACCEPT:ACCEPTABLE_PRICE:|COUNTER:COUNTER_OFFER:\d+|REJECT:(ABOVE_WALKAWAY|BLACKLISTED_SUPPLIER):|"
                 r"ESCALATE:(MISSING_INFO|PROMPT_INJECTION|BANNED_TERM|OVER_APPROVAL_LIMIT):)$")
assert [r["thread_id"] for r in sub] == ids, "thread_ids differ from threads.csv"
bad = [r for r in sub if not pat.match(r["label"])]
assert not bad, bad[:5]
print(f"OK: {len(sub)} rows, all labels well-formed")
for k, v in Counter(r["label"].rsplit(":", 1)[0] for r in sub).most_common():
    print(f"{v:5d}  {k}")